# 第4章 模版匹配

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

我们编程实现互相关公式，并测试下是否可以达到想要的模板匹配效果。

In [ ]:

import cv2
from matplotlib import pyplot as plt
import numpy as np

# 计算互相关
def CCORR(img, temp):
    w, h = temp.shape[::-1]
    W, H = img.shape[::-1]
    img = np.array(img, dtype='float')
    temp = np.array(temp, dtype='float')
    res = np.zeros((W - w + 1, H - h + 1))
    
    # 利用循环计算互相关的值
    for i in range(W - w + 1):
        for j in range(H - h + 1):
            res[i, j] = np.sum(temp * img[j:j + h, i:i + w])
    return res
    
# 构建单目标匹配类
class temp_match_single():
    def __init__(self, img, temp):
        self.img = img
        self.temp = temp
        
    def match(self):
        # 输入目标图像
        img = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)
        # 输入模板图像
        temp = cv2.cvtColor(self.temp, cv2.COLOR_BGR2GRAY)
        w, h = temp.shape[::-1]
        
        # 计算互相关
        res = CCORR(img, temp)
        
        # 找到互相关值最大的位置
        loc = np.where(res == np.max(res))
        top_left = [int(loc[0]), int(loc[1])]
        bottom_right = (top_left[0] + w, top_left[1] + h)
        
        # 将其框出
        cv2.rectangle(self.img, top_left, bottom_right, (255,255,255), 1)
        plot_image(self.img[:, :, ::-1], 'Matching Result by CCORR')

我们导入一张图像和一个模板，进行模板匹配。

In [ ]:
img = cv_imread('lena_tri.jpg')
template = cv_imread('lena_small.png')

test = temp_match_single(img, template)
plot_image(test.temp[:, :, ::-1], 'Template Image')
plot_image(test.img[:, :, ::-1], 'Target Image')

test.match()

结果有没有出乎大家的意料？我们并没有匹配到正确目标！问题出在哪里呢？给大家一点时间思考。

大家仔细观察互相关的公式，其结果是模板图像像素值和输入图像子图像素值直接相乘，那么在模板一定的情况下，子图整体像素值越大（也就是越亮），那么它们之间的乘积也就越大。所以互相关更容易找到输入图像中亮度高的区域，而不是和模板最相似的区域。那么如何解决这个问题呢？显然我们需要该相似度度量与输入图像中每个子图整体像素值大小无关，这就需要标准化。接下来我们引入另一种度量指标——标准化互相关。

#### 4.2.2.2 标准化互相关

再编程实现标准化互相关，这里重写CCORR()函数，再试试是否可以找到想要的目标区域。

In [ ]:
# 对原有CCORR()函数进行改写，定义标准化互相关
def CCORR(img, temp, normalize=True):
    w, h = temp.shape[::-1]
    W, H = img.shape[::-1]
    res = np.zeros((W-w+1, H-h+1))
    img = np.array(img, dtype='float')
    temp = np.array(temp, dtype='float')
    t = np.sqrt(np.sum(temp**2))
    for i in range(W-w+1):
        for j in range(H-h+1):
            res[i,j] = np.sum(temp*img[j:j+h, i:i+w])
            # 在这里进行归一化操作
            if normalize:
                res[i,j] = res[i,j] / t / np.sqrt(np.sum(img[j:j+h, i:i+w]**2)) 
    return res
    
img = cv_imread('lena_tri.jpg')
template = cv_imread('lena_small.png')

test = temp_match_single(img, template)

test.match()

由结果可见，我们通过标准化，消除了输入图像子图整体亮度变化对相似度计算的影响，提升了互相关度量的鲁棒性。

接下来，将实现多目标模板匹配。
为了方便，我们将直接使用标准化互相关作为度量指标，并直接调用库函数cv2.TM_CCORR_NORMED()来完成标准化互相关。

In [ ]:
import numpy as np

# 利用快速排序算法找到数列中第k大的值
def findKth(s, k):
    return findKth_c(s, 0, len(s) - 1, k)

def findKth_c(s, low, high, k):
    m = partition(s, low, high)
    if m == len(s) - k:
        return s[m]
    elif m < len(s) - k:
        return findKth_c(s, m + 1, high, k)
    else:
        return findKth_c(s, low, m - 1, k)

def partition(s, low, high):
    pivot, j = s[low], low
    for i in range(low + 1, high + 1):
        if s[i] <= pivot:
            j += 1
            s[i], s[j] = s[j], s[i]
    s[j], s[low] = s[low], s[j]
    return j

# 构建多目标模板匹配类
class temp_match_multi():
    def __init__(self, img, temp, k=50):
        self.img = img
        self.temp = temp
        # 定义需要匹配的个数
        self.k = k
        
    def match(self):
        # 输入模板图像和目标图像
        temp = cv2.cvtColor(self.temp, cv2.COLOR_BGR2GRAY)
        img = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)

        w, h = temp.shape[::-1]
        method = eval('cv2.TM_CCORR_NORMED')
        res = cv2.matchTemplate(img, temp, method)
        temp = list(np.array(res).flatten())
        
        # 寻找res中第k大的值
        threshold = findKth(temp, self.k+1)
        print('设定的阈值为：', threshold)
        loc = np.where(res >= threshold)
        
        # 将找到的子图框选出
        for pt in zip(*loc[::-1]):
            cv2.rectangle(self.img, pt, (pt[0] + w, pt[1] + h), (255,255,255), 1)
            plt.imshow(self.img[:, :, ::-1]), plt.xticks([]), plt.yticks([])
        plt.show()

In [ ]:
img = cv_imread('img.png')
template = cv_imread('temp.png')

test_multi = temp_match_multi(img, template, k=50)
plot_image(test_multi.temp[:, :, ::-1], 'Template Image')
plot_image(test_multi.img[:, :, ::-1], 'Target Image')
test_multi.match()


---

## 📝 
练习：手写模板匹配



**练习目标**：手写实现模板匹配算法。

**要求**：
1. 实现 `template_match_manual(image, template, method)` 函数
2. 支持 SAD、SSD、NCC 三种匹配方法
3. 在结果图上标注匹配位置
4. 与 cv2.matchTemplate 结果对比


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def template_match_manual(image, template, method='ncc'):
    """
    手写模板匹配
    - image: 搜索图像 (H, W)
    - template: 模板 (h, w)
    - method: 'sad' (绝对差和), 'ssd' (平方差和), 'ncc' (归一化互相关)
    """
    img_h, img_w = image.shape
    tpl_h, tpl_w = template.shape
    
    # 结果图尺寸（valid 模式）
    result_h = img_h - tpl_h + 1
    result_w = img_w - tpl_w + 1
    result = np.zeros((result_h, result_w), dtype=np.float32)
    
    # 转换为 float
    img = image.astype(np.float32)
    tpl = template.astype(np.float32)
    tpl_mean = tpl.mean()
    tpl_std = tpl.std() + 1e-6
    
    for y in range(result_h):
        for x in range(result_w):
            region = img[y:y+tpl_h, x:x+tpl_w]
            
            if method == 'sad':
                # Sum of Absolute Differences
                result[y, x] = -np.sum(np.abs(region - tpl))  # 取负值表示越大越好
            elif method == 'ssd':
                # Sum of Squared Differences
                result[y, x] = -np.sum((region - tpl)**2)
            elif method == 'ncc':
                # Normalized Cross-Correlation
                region_mean = region.mean()
                region_std = region.std() + 1e-6
                ncc = np.sum((region - region_mean) * (tpl - tpl_mean)) / (region_std * tpl_std)
                result[y, x] = ncc
    
    return result

# 测试
img = cv_imread('img.png', cv2.IMREAD_GRAYSCALE)
tpl = cv_imread('temp.png', cv2.IMREAD_GRAYSCALE)

# 手写匹配
result_manual = template_match_manual(img, tpl, 'ncc')
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result_manual)

# OpenCV 匹配
result_cv = cv2.matchTemplate(img, tpl, cv2.TM_CCOEFF_NORMED)
_, max_val_cv, _, max_loc_cv = cv2.minMaxLoc(result_cv)

print(f"手写匹配 - 最佳位置: {max_loc}, 最大值: {max_val:.4f}")
print(f"OpenCV匹配 - 最佳位置: {max_loc_cv}, 最大值: {max_val_cv:.4f}")
print(f"位置一致: {max_loc == max_loc_cv}")

# 可视化
img_color = cv_imread('img.png')
tpl_h, tpl_w = tpl.shape
cv2.rectangle(img_color, max_loc, (max_loc[0] + tpl_w, max_loc[1] + tpl_h), (0, 255, 0), 2)
cv_imwrite('template_match_result.jpg', img_color)
print("结果已保存到 template_match_result.jpg")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
